In [0]:
from pyspark.sql import functions as F, Window

# Ler da tabela Silver
df_silver = spark.table("b3_pipeline.silver_b3_stocks")

# ── Agregação 1: Ranking de retorno acumulado ──
w_last = Window.partitionBy("ticker").orderBy(F.desc("date"))

gold_retorno = df_silver \
    .withColumn("rn", F.row_number().over(w_last)) \
    .filter(F.col("rn") == 1) \
    .select("ticker", "setor", "date",
            "close", "cumulative_return_pct") \
    .orderBy(F.desc("cumulative_return_pct"))

# ── Agregação 2: Volatilidade por setor ──
gold_volatilidade = df_silver.groupBy("setor") \
    .agg(
        F.stddev("daily_return_pct").alias("volatilidade"),
        F.avg("daily_return_pct").alias("retorno_medio_diario"),
        F.count("*").alias("total_registros")
    ).orderBy(F.desc("volatilidade"))

# ── Salvar Gold ──
gold_retorno.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("b3_pipeline.gold_ranking_retorno")

gold_volatilidade.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("b3_pipeline.gold_volatilidade_setor")

print("✅ Gold salvo com sucesso!")
print("\n📊 Ranking de retorno acumulado:")
display(gold_retorno)
print("\n📊 Volatilidade por setor:")
display(gold_volatilidade)

✅ Gold salvo com sucesso!

📊 Ranking de retorno acumulado:


ticker,setor,date,close,cumulative_return_pct
PETR4,commodities,2026-06-05,40.88999938964844,310.060594398552
ITUB4,bancos,2026-06-05,38.83000183105469,174.2753458198619
BBAS3,bancos,2026-06-05,19.170000076293945,87.64674895309177
VIVT3,telecom,2026-06-05,32.95000076293945,81.99460420077583
BRAP4,commodities,2026-06-05,21.8799991607666,50.98255156675666
VALE3,commodities,2026-06-05,78.69999694824219,48.40145723190101
BBDC4,bancos,2026-06-05,17.469999313354492,35.43023997352701
TOTS3,tech,2026-06-05,33.099998474121094,28.045421845428486
LREN3,varejo,2026-06-05,14.890000343322754,-14.518948422942103
MGLU3,varejo,2026-06-05,5.440000057220459,-90.55082482796276



📊 Volatilidade por setor:


setor,volatilidade,retorno_medio_diario,total_registros
varejo,3.694033757005557,-0.0460986276904646,2206
tech,2.2310049018300773,0.0474345676579548,1103
commodities,1.8293256129679694,0.08375452309307975,3309
bancos,1.6663327291180314,0.07266722694328759,3309
telecom,1.4330097703039315,0.06457540784551138,1103
